# Super-Wised Coding LLM 1.0 — Kaggle Free GPU Smoke Test

This notebook verifies the complete **from-scratch Batch 1 coding LLM pipeline** (JavaScript, TypeScript, Node.js, Express.js, and NestJS) on a **Kaggle free GPU (T4 / P100)** or CPU using `configs/model.json`, `configs/training.json`, and `configs/data.json`:
- Custom decoder-only Transformer (`src/transformer/`) with **RoPE**, **RMSNorm**, **SwiGLU**, **Grouped-Query Attention (GQA)**, and **causal self-attention** initialized with random weights.
- Uses the `bigcode/starcoder2-15b` **tokenizer only** (`src/tokenizer/`; no pretrained model weights are ever loaded).
- End-to-end verification: licensed Batch 1 data filtering, deduplication, secret detection, tokenization, binary dataset validation, training, automatic checkpoint resume, evaluation, and SafeTensors release packaging.

> **Important Hardware Limit:** Training a 10B model from scratch is **not** possible on a free Kaggle GPU session (constrained by 16 GB VRAM and session wall-clock limits).

In [ ]:
import glob
import os
import pathlib
import shutil
import zipfile
import torch

def setup_kaggle_workspace() -> pathlib.Path:
    cwd = pathlib.Path.cwd()
    if (cwd / 'pyproject.toml').exists() and (cwd / 'src' / 'transformer' / 'model.py').exists():
        return cwd
    if (cwd.parent / 'pyproject.toml').exists() and (cwd.parent / 'src' / 'transformer' / 'model.py').exists():
        os.chdir(cwd.parent)
        return cwd.parent

    working_root = pathlib.Path('/kaggle/working/super-wised-code-llm-1.0')
    if (working_root / 'pyproject.toml').exists():
        os.chdir(working_root)
        return working_root

    for marker in glob.glob('/kaggle/input/**/pyproject.toml', recursive=True):
        src_dir = pathlib.Path(marker).parent
        if (src_dir / 'src' / 'transformer' / 'model.py').exists():
            shutil.copytree(src_dir, working_root, dirs_exist_ok=True)
            os.chdir(working_root)
            return working_root

    zips = glob.glob('/kaggle/input/**/*.zip', recursive=True)
    if zips:
        extract_root = pathlib.Path('/kaggle/working')
        with zipfile.ZipFile(zips[0]) as zf:
            zf.extractall(extract_root)
        for marker in glob.glob('/kaggle/working/**/pyproject.toml', recursive=True):
            found = pathlib.Path(marker).parent
            os.chdir(found)
            return found

    raise FileNotFoundError(
        'Attach super-wised-code-llm-1.0 as a Kaggle dataset/zip input or run from the repository root.'
    )

project_dir = setup_kaggle_workspace()
print('Working directory:', project_dir)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU device:', torch.cuda.get_device_name(0))

## 1. Verify Dependencies

In [ ]:
import importlib.util
import subprocess
import sys

required_modules = ['torch', 'transformers', 'tokenizers', 'huggingface_hub', 'datasets', 'numpy', 'safetensors']
missing = [m for m in required_modules if importlib.util.find_spec(m) is None]
if missing:
    print('Installing missing modules:', missing)
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'])
else:
    print('All required modules are installed.')

## 2. Run Unit Test Suite

In [ ]:
!python -m unittest discover -s tests -v

## 3. Prepare and Validate Batch 1 Dataset (JS / TS / Node.js / Express.js / NestJS)

In [ ]:
!python scripts/prepare_data.py --generate-sample-jsonl --sample-records 240 --input-jsonl data/raw/raw.jsonl --output-dir data/tokenized --tokenizer bigcode/starcoder2-15b --min-split-tokens 300
!python scripts/prepare_data.py --validate-only --output-dir data/tokenized --min-split-tokens 300

## 4. Train Small Configurable Model and Verify Automatic Checkpoint Resume

In [ ]:
!python scripts/train.py --config configs/model.json --training-config configs/training.json --data-dir data/tokenized --output-dir checkpoints --log-file logs/metrics.jsonl --no-resume --max-steps 30

In [ ]:
!python scripts/train.py --config configs/model.json --training-config configs/training.json --data-dir data/tokenized --output-dir checkpoints --log-file logs/metrics.jsonl --resume --max-steps 50

## 5. Evaluate Validation Perplexity and Batch 1 Code Generation

In [ ]:
!python scripts/evaluate.py --config configs/model.json --data-dir data/tokenized --checkpoint checkpoints/best.pt --output-json logs/eval_report.json

## 6. Package Release Artifacts

In [ ]:
!python scripts/publish.py --checkpoint checkpoints/best.pt --config configs/model.json --repo-id YOUR_HF_USERNAME/super-wised-code-llm-1.0 --output-dir release